## Dataset

Models are trained on **PlantSeg** — A Large-Scale In-the-wild Dataset for Plant Disease Segmentation (11,400+ images, 115 disease classes).

- Paper: https://arxiv.org/abs/2409.04038
- Download (Zenodo): https://zenodo.org/records/14935094
- Project: https://github.com/tqwei05/PlantSeg

> Wei, T., Chen, Z., Yu, X., Chapman, S., Melloy, P., Huang, Z. "PlantSeg: A Large-Scale In-the-wild Dataset for Plant Disease Segmentation." arXiv preprint arXiv:2409.04038, 2024.

The `plantsegv3/` folder in this notebook is the PlantSeg dataset (COCO annotations converted to YOLO label format).


In [1]:
import os
import time
from collections import defaultdict
import json

import torch
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.transforms import functional as F
from torchvision.datasets import CocoDetection
from torch.utils.data import DataLoader

In [2]:
# ─── Paths ───────────────────────────────────────────────────────────
COCO_ROOT    = "/home/sagemaker-user/plantsegv3"
DATA_YAML    = os.path.join(COCO_ROOT, "data.yaml")

# RF‑DETR expects subfolders train/ valid/ test/ each with its own
# _annotations.coco.json + images.
RFDS_ROOT    = os.path.join(COCO_ROOT, "masks")
RF_TRAIN_ANN = os.path.join(RFDS_ROOT, "annotation_train.json")
RF_VAL_ANN   = os.path.join(RFDS_ROOT, "annotation_val.json")
RF_TEST_ANN  = os.path.join(RFDS_ROOT, "annotation_test.json")
RF_TRAIN_IMG = os.path.join(COCO_ROOT, "images", "train")
RF_VAL_IMG   = os.path.join(COCO_ROOT, "images", "val")
RF_TEST_IMG  = os.path.join(COCO_ROOT, "images", "test")

# Store results for each model
results = defaultdict(dict)


In [3]:
# !git clone https://github.com/ultralytics/yolov5  # clone the ultralytics repo
# %cd yolov5
# !pip install -r requirements.txt

In [ ]:
model = YOLO('yolov8l.pt')  # or 'yolov8l.pt' for pretrained
model.train(data='plantsegv3/data.yaml', epochs=300, imgsz=416, batch=16, save_period=7, patience=20)

Ultralytics 8.3.135 🚀 Python-3.12.9 torch-2.5.1 CUDA:0 (Tesla T4, 14918MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=plantsegv3/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train16, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20, perspective=0.0, plots=True, pose=12.0, pretrained=True, profil

train: Scanning /home/sagemaker-user/plantsegv3/labels/train.cache... 7916 images, 0 backgrounds, 44 corrupt: 100%|██████████| 7916/7916 [00:00<?, ?it/s]

train: /home/sagemaker-user/plantsegv3/images/train/apple_black_rot_1.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0187]
train: /home/sagemaker-user/plantsegv3/images/train/apple_black_rot_6.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1912      1.1163      1.0026]
train: /home/sagemaker-user/plantsegv3/images/train/apple_mosaic_virus_4.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     2.3204      1.0991      1.5287      1.6011      1.3444       1.082]
train: /home/sagemaker-user/plantsegv3/images/train/bean_rust_2.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.2954       1.314]
train: /home/sagemaker-user/plantsegv3/images/train/celery_anthracnose_google_0001.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.2538      1.2235      1.1094]
train: /home/sagemaker-user/plantsegv3/images/train/cit

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 801.4±530.9 MB/s, size: 60.4 KB)


val: Scanning /home/sagemaker-user/plantsegv3/labels/val.cache... 1247 images, 0 backgrounds, 1 corrupt: 100%|██████████| 1247/1247 [00:00<?, ?it/s]

val: /home/sagemaker-user/plantsegv3/images/val/tomato_leaf_mold_3.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0862]


Plotting labels to runs/detect/train16/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.0005), 103 bias(decay=0.0)


2025/05/16 05:45:39 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.


MLflow: logging run_id(2ac901cace4345a6a49a01dc592c4520) to runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 416 train, 416 val
Using 4 dataloader workers
Logging results to runs/detect/train16
Starting training for 300 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/300      4.26G      1.804      5.619      1.713        165        416:  14%|█▎        | 67/492 [00:53<05:36,  1.26it/s]

In [51]:
model.val()

Ultralytics 8.3.133 🚀 Python-3.12.9 torch-2.5.1 CUDA:0 (Tesla T4, 14918MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1732.5±762.2 MB/s, size: 381.8 KB)


val: Scanning /home/sagemaker-user/plantsegv3/labels/val.cache... 1247 images, 0 backgrounds, 1 corrupt: 100%|██████████| 1247/1247 [00:00<?, ?it/s]

val: /home/sagemaker-user/plantsegv3/images/val/tomato_leaf_mold_3.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0862]



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 78/78 [00:19<00:00,  3.95it/s]


                   all       1246       8919      0.419      0.371      0.352      0.223
                     0          9         51      0.432      0.392       0.39      0.196
                     1         10         63       0.44      0.397       0.39      0.229
                     2         15         98      0.533      0.459      0.506      0.336
                     3         27        283      0.467      0.369      0.345      0.169
                     4          8        104      0.516      0.524      0.479      0.238
                     5         18         54      0.515      0.333      0.381      0.281
                     6         17         62      0.269      0.355      0.241       0.13
                     7          7         34      0.535      0.706      0.597      0.486
                     8          6         35      0.393      0.457      0.392      0.249
                     9          7         28      0.342        0.5      0.403      0.193
                    1

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  59,  60,  61,
        62,  63,  64,  65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fd9d47c1e80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.00

In [56]:
#progress for above cell saved in runs/detect/train6/weights/best.pt

In [19]:
#eight_x_model = YOLO('yolov8x.pt')  # or 'yolov8l.pt' for pretrained
eight_x_model = YOLO('runs/detect/train13/weights/last.pt')  # cont from epoch 40
eight_x_model.train(data='plantsegv3/data.yaml', epochs=1000, imgsz=416, batch=16, save_period=5 , patience=30)

New https://pypi.org/project/ultralytics/8.3.134 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.133 🚀 Python-3.12.9 torch-2.5.1 CUDA:0 (Tesla T4, 14918MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=plantsegv3/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1000, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/train13/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train14, nbs=64, nms=False, opset=None,

train: Scanning /home/sagemaker-user/plantsegv3/labels/train.cache... 7916 images, 0 backgrounds, 44 corrupt: 100%|██████████| 7916/7916 [00:00<?, ?it/s]

train: /home/sagemaker-user/plantsegv3/images/train/apple_black_rot_1.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0187]
train: /home/sagemaker-user/plantsegv3/images/train/apple_black_rot_6.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1912      1.1163      1.0026]
train: /home/sagemaker-user/plantsegv3/images/train/apple_mosaic_virus_4.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     2.3204      1.0991      1.5287      1.6011      1.3444       1.082]
train: /home/sagemaker-user/plantsegv3/images/train/bean_rust_2.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.2954       1.314]
train: /home/sagemaker-user/plantsegv3/images/train/celery_anthracnose_google_0001.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.2538      1.2235      1.1094]
train: /home/sagemaker-user/plantsegv3/images/train/cit

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 625.0±209.2 MB/s, size: 60.4 KB)


val: Scanning /home/sagemaker-user/plantsegv3/labels/val.cache... 1247 images, 0 backgrounds, 1 corrupt: 100%|██████████| 1247/1247 [00:00<?, ?it/s]

...

    32/1000      12.5G      1.145      1.223      1.128        122        416: 100%|██████████| 492/492 [04:07<00:00,  1.99it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:14<00:00,  2.69it/s]


                   all       1246       8919      0.367      0.333      0.309      0.195
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 2, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

32 epochs completed in 2.350 hours.
Optimizer stripped from runs/detect/train14/weights/last.pt, 136.9MB
Optimizer stripped from runs/detect/train14/weights/best.pt, 136.9MB

Validating runs/detect/train14/weights/best.pt...
Ultralytics 8.3.133 🚀 Python-3.12.9 torch-2.5.1 CUDA:0 (Tesla T4, 14918MiB)
Model summary (fused): 112 layers, 68,234,313 parameters, 0 gradients, 258.0 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 39/39 [00:14<00:00,  2.64it/s]


                   all       1246       8919      0.358      0.369      0.334      0.211
                     0          9         51      0.422      0.429      0.335      0.189
                     1         10         63      0.313      0.397      0.319      0.192
                     2         15         98      0.428      0.561      0.528      0.324
                     3         27        283      0.415      0.385      0.332       0.15
                     4          8        104      0.415      0.423      0.373      0.193
                     5         18         54      0.605      0.312      0.406      0.301
                     6         17         62      0.273      0.323      0.212      0.107
                     7          7         34      0.381      0.735      0.646      0.511
                     8          6         35      0.571      0.314      0.428      0.268
                     9          7         28      0.294        0.5      0.445      0.269
                    1

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  59,  60,  61,
        62,  63,  64,  65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f76e261ade0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.00

In [ ]:
#runs/detect/train14

In [ ]:
runs/detect/train9

In [4]:
def train_yolov10():
    model = YOLO("yolov10n.pt")
    model.train(data=DATA_YAML, epochs=10)  # short smoke-test
    return model

# ─── 4. Eval YOLOv10 ────────────────────────────────────────────────
def eval_yolov10(model):
    start = time.time()
    metrics = model.val(data=DATA_YAML, split="test")
    results["YOLOv10"]["latency_sec"] = time.time() - start
    results["YOLOv10"]["metrics"]     = metrics.box


In [1]:
# YOLOv8
yolo8_model = train_yolov8()
eval_yolov8(yolo8_model)

NameError: name 'train_yolov8' is not defined

In [3]:
yolo8_model.val()

NameError: name 'yolo8_model' is not defined

In [14]:
%cd

/home/sagemaker-user


In [2]:
# Load your trained model
model = YOLO("runs/detect/train182")

# Run inference on a folder of test images
results = model("plantsegv3/images/test", save=True, save_txt=True)

# Optional: print results summary
for r in results:
    print(r.path)            # file path
    print(r.boxes.xyxy)      # bounding box coordinates
    print(r.boxes.conf)      # confidence scores
    print(r.boxes.cls)       # predicted class indices


NameError: name 'YOLO' is not defined

In [46]:
from PIL import Image
from torch.utils.data import Dataset

class CocoDetectionDataset(Dataset):
    def __init__(self, image_dir, annotation_path, transforms=None):
        self.image_dir = image_dir
        self.coco = COCO(annotation_path)
        self.ids = sorted(self.coco.imgs.keys())
        self.transforms = transforms

    def __getitem__(self, index):
        coco = self.coco
        img_id = self.ids[index]
        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)
        path = coco.loadImgs(img_id)[0]['file_name']
        img = Image.open(os.path.join(self.image_dir, path)).convert("RGB")

        boxes, labels = [], []
        for ann in anns:
            if ann.get('iscrowd', 0) == 0 and 'bbox' in ann and 'category_id' in ann:
                x, y, w, h = ann['bbox']
                if w > 0 and h > 0:
                    boxes.append([x, y, x + w, y + h])
                    labels.append(ann['category_id'])

        boxes = torch.as_tensor(boxes, dtype=torch.float32) if boxes else torch.empty((0, 4), dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64) if labels else torch.empty((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor(img_id),
        }

        if self.transforms:
            if hasattr(self.transforms, '__call__'):
                if hasattr(self.transforms, 'transforms') and len(self.transforms.transforms) > 1:
                    img, target = self.transforms(img, target)
                else:
                    img = self.transforms(img)
        else:
            img = T.ToTensor()(img)

        return img, target

    def __len__(self):
        return len(self.ids)


In [47]:
def save_checkpoint(epoch, model, optimizer, best_metric, epochs_without_improvement,
                    checkpoint_path, val_metrics=None, epoch_losses=None, iteration_losses=None,
                    metrics_json_path="training_metrics.json"):
    # Save model checkpoint
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimiser_state_dict': optimizer.state_dict(),
        'best_val_metric': best_metric,
        'epochs_without_improvement': epochs_without_improvement
    }
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved to {checkpoint_path}")

    # Save metrics to JSON
    all_metrics = []
    if os.path.exists(metrics_json_path):
        with open(metrics_json_path, "r") as f:
            all_metrics = json.load(f)

    all_metrics.append({
        "epoch": epoch + 1,
        "train_loss": epoch_losses[-1] if epoch_losses else None,
        "val_loss": val_metrics[-1] if val_metrics else None
    })

    with open(metrics_json_path, "w") as f:
        json.dump(all_metrics, f, indent=2)
    print(f"Metrics written to {metrics_json_path}")


In [48]:
def load_checkpoint(model, optimizer, checkpoint_path, metrics_json_path="training_metrics.json"):
    if not os.path.exists(checkpoint_path):
        print(f"No checkpoint found at {checkpoint_path}. Starting training from scratch.")
        return model, optimizer, 0, float('inf'), 0, [], [], []

    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimiser_state_dict'])

    val_metrics, epoch_losses = [], []
    if os.path.exists(metrics_json_path):
        with open(metrics_json_path, "r") as f:
            data = json.load(f)
            val_metrics = [entry.get("val_loss", 0.0) for entry in data]
            epoch_losses = [entry.get("train_loss", 0.0) for entry in data]

    return (
        model,
        optimizer,
        checkpoint.get('epoch', 0) + 1,
        checkpoint.get('best_val_metric', float('inf')),
        checkpoint.get('epochs_without_improvement', 0),
        val_metrics,
        epoch_losses,
        []
    )

In [49]:
import os
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader
import torchvision.transforms as T

# Dataset and transforms
my_transforms = T.Compose([T.ToTensor()])

train_dataset = CocoDetectionDataset(
    image_dir='plantsegv3/images/train',
    annotation_path='plantsegv3/masks/annotation_train.json',
    transforms=my_transforms
)

validation_dataset = CocoDetectionDataset(
    image_dir='plantsegv3/images/val',
    annotation_path='plantsegv3/masks/annotation_val.json',
    transforms=my_transforms
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
validation_loader = DataLoader(validation_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# Model
model = fasterrcnn_resnet50_fpn(pretrained=True)
num_classes = 115
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optimizer and scheduler
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Checkpoint loading
checkpoint_file = "checkpossssint.pth"
if os.path.exists(checkpoint_file):
    print(f"Loading checkpoint from {checkpoint_file}...")
    model, optimizer, start_epoch, best_val_metric, epochs_without_improvement, val_metrics, epoch_train_losses, iteration_train_losses = load_checkpoint(
        model, optimizer, checkpoint_file
    )
else:
    print("No checkpoint found. Starting from scratch.")
    start_epoch = 0
    best_val_metric = 0
    epochs_without_improvement = 0
    val_metrics = []
    epoch_train_losses = []
    iteration_train_losses = []

# Early stopping setup
patience = 35
early_stop = False


loading annotations into memory...
Done (t=0.51s)
creating index...
index created!
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!


/opt/conda/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


No checkpoint found. Starting from scratch.


In [51]:
import time

num_epochs = 1000  # Set high enough; early stopping will halt if needed

for epoch in range(start_epoch, num_epochs):
    print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")
    model.train()
    epoch_loss_sum = 0.0
    start_time = time.time()

    # Training Loop
    for batch_idx, (images, targets) in enumerate(train_loader):
        print(f"  Batch {batch_idx + 1}/{len(train_loader)}...")

        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss_sum += loss.item()

    lr_scheduler.step()
    train_avg_loss = epoch_loss_sum / len(train_loader)
    epoch_train_losses.append(train_avg_loss)
    print(f"Epoch {epoch + 1} training complete. Avg loss: {train_avg_loss:.4f}")

    # Validation Loop
    model.train()  # Still in train mode to get loss
    val_loss_sum = 0.0
    with torch.no_grad():
        for images, targets in validation_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
    
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            val_loss_sum += loss.item()

    val_avg_loss = val_loss_sum / len(validation_loader)
    val_metrics.append(val_avg_loss)
    print(f"Validation avg loss: {val_avg_loss:.4f} | Time: {time.time() - start_time:.2f}s")

    metrics_data = [
        {"val_loss": v, "train_loss": t}
        for v, t in zip(val_metrics, epoch_train_losses)
    ]
    
    with open("training_metrics.json", "w") as f:
        json.dump(metrics_data, f, indent=2)

    # Early Stopping Check
    if val_avg_loss < best_val_metric:
        print(f"New best validation loss: {val_avg_loss:.4f} (prev: {best_val_metric:.4f})")
        best_val_metric = val_avg_loss
        epochs_without_improvement = 0
        save_checkpoint(
            epoch, model, optimizer, best_val_metric, epochs_without_improvement,
            checkpoint_file, val_metrics, epoch_train_losses, iteration_train_losses
        )
    else:
        epochs_without_improvement += 1
        print(f"No improvement. Patience: {epochs_without_improvement}/{patience}")

        if epochs_without_improvement >= patience:
            save_checkpoint(
                epoch, model, optimizer, best_val_metric, epochs_without_improvement,
                checkpoint_file, val_metrics, epoch_train_losses, iteration_train_losses
            )
            print("Early stopping triggered.")
            break



--- Epoch 1/1000 ---
  Batch 1/1979...
  Batch 2/1979...
  Batch 3/1979...
  Batch 4/1979...
  Batch 5/1979...
  Batch 6/1979...
  Batch 7/1979...
  Batch 8/1979...
  Batch 9/1979...
  Batch 10/1979...
  Batch 11/1979...
  Batch 12/1979...
  Batch 13/1979...
  Batch 14/1979...
  Batch 15/1979...
  Batch 16/1979...
  Batch 17/1979...
  Batch 18/1979...
  Batch 19/1979...
  Batch 20/1979...
  Batch 21/1979...
  Batch 22/1979...
  Batch 23/1979...
  Batch 24/1979...
  Batch 25/1979...
  Batch 26/1979...
  Batch 27/1979...
  Batch 28/1979...
  Batch 29/1979...
  Batch 30/1979...
  Batch 31/1979...
  Batch 32/1979...
  Batch 33/1979...
  Batch 34/1979...
  Batch 35/1979...
  Batch 36/1979...
  Batch 37/1979...
  Batch 38/1979...
  Batch 39/1979...
  Batch 40/1979...
  Batch 41/1979...
  Batch 42/1979...
  Batch 43/1979...
  Batch 44/1979...
  Batch 45/1979...
  Batch 46/1979...
  Batch 47/1979...
  Batch 48/1979...
  Batch 49/1979...
  Batch 50/1979...
  Batch 51/1979...
  Batch 52/1979...

KeyboardInterrupt: 

In [ ]:
model = fasterrcnn_resnet50_fpn(pretrained=False)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Load weights
checkpoint = torch.load("checkpoint.pth", map_location=torch.device("cpu"))  # or 'cuda' if available
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
model.to(device)

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import torch
import os
import json
from tqdm import tqdm

def evaluate_model(model, data_loader, device, annotation_path, iou_type="bbox", output_json="detections.json"):
    model.eval()
    coco_gt = COCO(annotation_path)
    coco_results = []

    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="Evaluating"):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):
                image_id = target["image_id"].item()
                boxes = output["boxes"].cpu().numpy()
                scores = output["scores"].cpu().numpy()
                labels = output["labels"].cpu().numpy()
                
                for box, score, label in zip(boxes, scores, labels):
                    x_min, y_min, x_max, y_max = box
                    width = x_max - x_min
                    height = y_max - y_min
                    coco_results.append({
                        "image_id": int(image_id),
                        "category_id": int(label),
                        "bbox": [float(x_min), float(y_min), float(width), float(height)],
                        "score": float(score)
                    })

    # Save detections to JSON
    with open(output_json, "w") as f:
        json.dump(coco_results, f, indent=2)

    # Load results and evaluate
    coco_dt = coco_gt.loadRes(output_json)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType=iou_type)
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return coco_eval.stats  # [AP, AP50, AP75, AP_small, AP_medium, AP_large, ...]


In [ ]:
from torch.utils.data import DataLoader

test_dataset = CocoDetectionDataset(
    image_dir='plantsegv3/images/test',
    annotation_path='plantsegv3/masks/annotation_test.json',
    transforms=T.Compose([T.ToTensor()])
)

test_loader = DataLoader(
    test_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x))
)

# Evaluate
stats = evaluate_model(
    model=model,
    data_loader=test_loader,
    device=device,
    annotation_path='plantsegv3/masks/annotation_test.json'
)
print(f"COCO mAP Stats: {stats}")

In [37]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import torch
import json
from tqdm import tqdm

def evaluate_coco(model, data_loader, device, annotation_path, iou_type="bbox", output_json="detections.json"):
    model.eval()
    coco_gt = COCO(annotation_path)
    coco_results = []

    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="Evaluating"):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):
                image_id = target["image_id"].item()
                boxes = output["boxes"].cpu().numpy()
                scores = output["scores"].cpu().numpy()
                labels = output["labels"].cpu().numpy()

                for box, score, label in zip(boxes, scores, labels):
                    x_min, y_min, x_max, y_max = box
                    width = max(0, x_max - x_min)
                    height = max(0, y_max - y_min)

                    if width > 0 and height > 0:
                        coco_results.append({
                            "image_id": int(image_id),
                            "category_id": int(label),
                            "bbox": [float(x_min), float(y_min), float(width), float(height)],
                            "score": float(score)
                        })

    # Save predictions to JSON
    with open(output_json, "w") as f:
        json.dump(coco_results, f)

    # Load predictions and evaluate
    coco_dt = coco_gt.loadRes(output_json)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType=iou_type)
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    # Extract key results into a dictionary
    stats = coco_eval.stats  # [AP, AP50, AP75, AP_small, AP_medium, AP_large, AR@1, AR@10, AR@100, AR@100 (large)]
    results = {
        "AP": stats[0],
        "AP50": stats[1],
        "AP75": stats[2],
        "AP_small": stats[3],
        "AP_medium": stats[4],
        "AP_large": stats[5],
        "AR@1": stats[6],
        "AR@10": stats[7],
        "AR@100": stats[8],
        "AR_large": stats[9],
    }

    return results
results = evaluate_coco(model, test_loader, device, plantsegv3/masks/annotation_test.json)

AttributeError: 'dict' object has no attribute 'pth'